In [ ]:
import numpy as np
import pandas as pd
import GPy
import copy
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge as ridge
from neurIPS.functions.data_simulation import data_simulation8
from neurIPS.functions.model_training import ICM

# -------------------------------
# Utility functions
# -------------------------------
rmse = lambda x, y: np.sqrt(mean_squared_error(x, y))

linear_reg_parameter_pairs = [
    (ridge(), {'alpha': np.logspace(-7, 7, 8), 'tol': [1e-3]})
]

def get_best_model_from_gridcv(X, Y, reg_parameter_pairs, print_flag=False, verbose=0):
    val_errs = []
    models = []
    x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2)
    for reg, parameters in reg_parameter_pairs:
        if print_flag:
            print('Trying', str(reg)[:50])
        model = GridSearchCV(reg, parameters, cv=2, refit=True, verbose=verbose, n_jobs=10)
        model.fit(X, Y)
        val_errs.append(rmse(y_test, model.predict(x_test)))
        models.append(copy.deepcopy(model.best_estimator_))
    min_ind = val_errs.index(min(val_errs))
    if verbose != 0:
        print(str(models[min_ind]), 'val rmse:{}'.format(val_errs[min_ind]))
    return copy.deepcopy(models[min_ind])

# -------------------------------
# GP-based Kallus adjustment
# -------------------------------
def run_methodGP(X_rct, Y_rct, T_rct, X_obs, Y_obs, T_obs, X_eval):
    # GP on observational data
    kern = GPy.kern.RBF(input_dim=1, variance=1., lengthscale=1.)
    f0_gp = GPy.models.GPRegression(np.vstack(X_obs[T_obs==0]), np.vstack(Y_obs[T_obs==0]), kern)
    f0_gp.optimize()
    f1_gp = GPy.models.GPRegression(np.vstack(X_obs[T_obs==1]), np.vstack(Y_obs[T_obs==1]), kern)
    f1_gp.optimize()

    # Predict CATE components
    omega_rct_pred = f1_gp.predict(np.vstack(X_rct))[0] - f0_gp.predict(np.vstack(X_rct))[0]
    omega_obs_pred = f1_gp.predict(np.vstack(X_obs))[0] - f0_gp.predict(np.vstack(X_obs))[0]
    omega_eval_pred = f1_gp.predict(np.vstack(X_eval))[0] - f0_gp.predict(np.vstack(X_eval))[0]

    # Estimate residual eta on RCT
    prop_rct = np.mean(T_rct)
    def ite_adjusted_outcome(Y, T, _propensity, c=None):
        c = np.mean(Y) if c is None else c
        _ite = np.zeros(Y.shape)
        _ite[T==1] = (Y[T==1] - c) / _propensity
        _ite[T==0] = -(Y[T==0] - c) / (1-_propensity)
        return _ite

    cate_rct_est = ite_adjusted_outcome(Y_rct, T_rct, _propensity=prop_rct)
    eta_rct_est = np.vstack(cate_rct_est) - np.vstack(omega_rct_pred)

    # Fit linear model to eta
    best_eta_model = get_best_model_from_gridcv(X_rct.reshape(-1,1), eta_rct_est, linear_reg_parameter_pairs)
    
    return copy.deepcopy(best_eta_model), omega_obs_pred, omega_eval_pred

# -------------------------------
# Initialize arrays to store results
# -------------------------------
num_simulations = 100
test_data = np.round(np.arange(-2.0, 2.04, 0.04), 2)

rmse_naiveGP_exp = np.zeros(num_simulations)
rmse_naiveGP_obs = np.zeros(num_simulations)
rmse_KallusGP = np.zeros(num_simulations)
rmse_ICM = np.zeros(num_simulations)

bias_naiveGP_exp = np.zeros(num_simulations)
bias_naiveGP_obs = np.zeros(num_simulations)
bias_KallusGP = np.zeros(num_simulations)
bias_ICM = np.zeros(num_simulations)

var_naiveGP_exp = np.zeros(num_simulations)
var_naiveGP_obs = np.zeros(num_simulations)
var_KallusGP = np.zeros(num_simulations)
var_ICM = np.zeros(num_simulations)

bias_naive_exp_df = pd.DataFrame(test_data)
bias_naive_obs_df = pd.DataFrame(test_data)
bias_kallus_df = pd.DataFrame(test_data)
bias_icm_df = pd.DataFrame(test_data)

all_data = []
best_rho = pd.read_csv('sim8_best_rho.csv').values[0][0]

# -------------------------------
# Simulation loop
# -------------------------------
for i in range(num_simulations):
    print(f"Simulation {i+1}")
    data_exp, data_obs = data_simulation8(1000, beta0=-3.0, beta1=-3.0)

    df_exp = pd.DataFrame(data_exp, columns=["S", "X", "A", "Y"])
    df_exp["datasetNo"] = i + 1
    df_obs = pd.DataFrame(data_obs, columns=["S", "X", "A", "Y"])
    df_obs["datasetNo"] = i + 1
    combined_df = pd.concat([df_exp, df_obs], ignore_index=True)
    all_data.append(combined_df)

    # Split by treatment group
    X0_obs, X1_obs = data_obs[:,1][data_obs[:,2]==0], data_obs[:,1][data_obs[:,2]==1]
    Y0_obs, Y1_obs = data_obs[:,3][data_obs[:,2]==0], data_obs[:,3][data_obs[:,2]==1]
    X0_exp, X1_exp = data_exp[:,1][data_exp[:,2]==0], data_exp[:,1][data_exp[:,2]==1]
    Y0_exp, Y1_exp = data_exp[:,3][data_exp[:,2]==0], data_exp[:,3][data_exp[:,2]==1]

    trueCATE = 1 + test_data + test_data**2

    # -------------------------------
    # Naive GP on observational data
    # -------------------------------
    kern = GPy.kern.RBF(input_dim=1, variance=1., lengthscale=1.)
    m0_obs = GPy.models.GPRegression(np.vstack(X0_obs), np.vstack(Y0_obs), kern)
    m0_obs.optimize()
    m1_obs = GPy.models.GPRegression(np.vstack(X1_obs), np.vstack(Y1_obs), kern)
    m1_obs.optimize()
    naiveCATE_obs = m1_obs.predict(np.vstack(test_data))[0] - m0_obs.predict(np.vstack(test_data))[0]

    # -------------------------------
    # Naive GP on experimental data
    # -------------------------------
    m0_exp = GPy.models.GPRegression(np.vstack(X0_exp), np.vstack(Y0_exp), kern)
    m0_exp.optimize()
    m1_exp = GPy.models.GPRegression(np.vstack(X1_exp), np.vstack(Y1_exp), kern)
    m1_exp.optimize()
    naiveCATE_exp = m1_exp.predict(np.vstack(test_data))[0] - m0_exp.predict(np.vstack(test_data))[0]

    # -------------------------------
    # Kallus GP
    # -------------------------------
    eta_model, omega_obs_pred, omega_eval_pred = run_methodGP(
        data_exp[:,1], data_exp[:,3], data_exp[:,2],
        data_obs[:,1], data_obs[:,3], data_obs[:,2],
        test_data
    )
    tau_pred = eta_model.predict(test_data.reshape(-1,1)) + omega_eval_pred

    # -------------------------------
    # ICM
    # -------------------------------
    m0_icm, m1_icm = ICM(X_E=data_exp[:,1], X_O=data_obs[:,1],
                          T_E=data_exp[:,2], T_O=data_obs[:,2],
                          Y_E=data_exp[:,3], Y_O=data_obs[:,3],
                          r=2, ID=1, AD=0, rho=best_rho)
    predCATE_icm = m1_icm.predict_noiseless(np.c_[np.vstack(test_data), np.zeros(test_data.shape[0])])[0] - \
                   m0_icm.predict_noiseless(np.c_[np.vstack(test_data), np.zeros(test_data.shape[0])])[0]

    # -------------------------------
    # Store metrics
    # -------------------------------
    rmse_naiveGP_exp[i] = rmse(naiveCATE_exp, trueCATE)
    rmse_naiveGP_obs[i] = rmse(naiveCATE_obs, trueCATE)
    rmse_KallusGP[i] = rmse(tau_pred, trueCATE)
    rmse_ICM[i] = rmse(predCATE_icm, trueCATE)

    bias_naiveGP_exp[i] = np.mean(naiveCATE_exp - trueCATE)
    bias_naiveGP_obs[i] = np.mean(naiveCATE_obs - trueCATE)
    bias_KallusGP[i] = np.mean(tau_pred - trueCATE)
    bias_ICM[i] = np.mean(predCATE_icm - trueCATE)

    var_naiveGP_exp[i] = np.var(naiveCATE_exp)
    var_naiveGP_obs[i] = np.var(naiveCATE_obs)
    var_KallusGP[i] = np.var(tau_pred)
    var_ICM[i] = np.var(predCATE_icm)

    bias_naive_exp_df[f"Iteration_{i+1}"] = naiveCATE_exp - trueCATE
    bias_naive_obs_df[f"Iteration_{i+1}"] = naiveCATE_obs - trueCATE
    bias_kallus_df[f"Iteration_{i+1}"] = tau_pred - trueCATE
    bias_icm_df[f"Iteration_{i+1}"] = predCATE_icm - trueCATE

# -------------------------------
# Save results
# -------------------------------
final_df_sim8 = pd.concat(all_data, ignore_index=True)
final_df_sim8.to_csv('datasets_sim8.csv', index=False)

sim8_modelComparison_RMSEs_df = pd.DataFrame(np.c_[rmse_naiveGP_exp, rmse_naiveGP_obs, rmse_KallusGP, rmse_ICM,
                                                   bias_naiveGP_exp, bias_naiveGP_obs, bias_KallusGP, bias_ICM,
                                                   var_naiveGP_exp, var_naiveGP_obs, var_KallusGP, var_ICM], 
                                            columns=['RMSE_naiveGPexp','RMSE_naiveGPobs','RMSE_KallusGP','RMSE_ICM',
                                                     'bias_naiveGPexp','bias_naiveGPobs','bias_KallusGP','bias_ICM',
                                                     'var_naiveGPexp','var_naiveGPobs','var_KallusGP','var_ICM'])
sim8_modelComparison_RMSEs_df.to_csv('sim8_modelComparison.csv', index=False)
bias_naive_exp_df.to_csv('bias_naive_exp.csv', index=False)
bias_naive_obs_df.to_csv('bias_naive_obs.csv', index=False)
bias_kallus_df.to_csv('bias_kallus.csv', index=False)
bias_icm_df.to_csv('bias_icm.csv', index=False)
